<strong><font size=18 color=pink>Ch3romaniiLearn</font></strong>

В данном ноутбуке я реализовал первую часть pet-проекта «Ch3romaniiLearn» - персонального ассистента для подготовки к экзаменам.<br>
Первая часть включает в себя сбор данных и подготовку датасета для дообучения модели, настройку дообучения и само дообучение, оценку результатов и выгрузку итоговой модели.


# **Настройка окружения**

In [1]:
!pip install -q unsloth
!pip install -q wandb tensorboard
!hf auth login
!hf auth whoami

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 727.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ch3romanii (ch3romanii-ch3romanii) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [23]:
%%time
%%capture
import os, numpy as np, torch, math

from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, TrainingArguments
from unsloth.chat_templates import train_on_responses_only

CPU times: user 868 µs, sys: 1.53 ms, total: 2.39 ms
Wall time: 3.86 ms


# **Загрузка и подготовка датасетов для дообучения**

Так как цель получить ассистенита для подготовки к экзаменам, то смыслом дообучения будет **научить модель приводить ученика к решению через наводящие вопросы**. Поэтому в качестве обучающего датасета нужны диалоги между учителем и учеником.<br>
Изначально я планировал найти полностью готовый датасет на HuggingFace и на его части дообучить модель, но хорошего подходящего варианта я не нашел, а сильно заморачиваться предобработкой я не хотел. Поэтому я сгенерировал 2000 диалогов с помощью `claude`, а затем отобрал 500 лучших уникальных диалогов. Дополнительно я частично использовал найденный датасет `mathdial` для того чтобы разбавить синтетику реальным диалогами. <br>
В итоге было использовано 650 диалогов, что достаточно для того, чтобы небольшая (1.5B) модель уловила нужный паттерн поведения.

In [5]:
sft_df1 = load_dataset('ptvnck/TutoringDialogs')
sft_df2 = load_dataset('eth-nlped/mathdial')
display(sft_df1)
print('='*50)
display(sft_df2)

README.md:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tutor_dialogues.jsonl:   0%|          | 0.00/707k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/7.18M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/599 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'category', 'subject', 'topic', 'style', 'num_turns', 'messages'],
        num_rows: 500
    })
})

DatasetDict({
    train: Dataset({
        features: ['qid', 'scenario', 'question', 'ground_truth', 'student_incorrect_solution', 'student_profile', 'teacher_described_confusion', 'self-correctness', 'self-typical-confusion', 'self-typical-interactions', 'conversation'],
        num_rows: 2262
    })
    test: Dataset({
        features: ['qid', 'scenario', 'question', 'ground_truth', 'student_incorrect_solution', 'student_profile', 'teacher_described_confusion', 'self-correctness', 'self-typical-confusion', 'self-typical-interactions', 'conversation'],
        num_rows: 599
    })
})

In [6]:
display(sft_df1['train'][0])
print('='*150)
display(sft_df2['train'][0])

{'id': 'tutor_dialogue_0001',
 'category': 'mixed_themes',
 'subject': ['computer_science'],
 'topic': 'problem_decomposition_multi_step',
 'style': {'teaching_style': 'scaffolded_hints',
  'difficulty': 'intermediate',
  'personality': None,
  'student_personality': None,
  'tutor_personality': None},
 'num_turns': 10,
 'messages': [{'role': 'student',
   'content': 'i need to write a program that checks if a word is a palindrome and i just froze staring at a blank editor'},
  {'role': 'tutor',
   'content': "Let's not write any code yet. In plain English, what does it mean for a word to be a palindrome?"},
  {'role': 'student', 'content': 'it reads the same forwards and backwards'},
  {'role': 'tutor',
   'content': "Given that definition, what two things could you compare to check if that's true?"},
  {'role': 'student',
   'content': 'the original word, and some reversed version of it'},
  {'role': 'tutor',
   'content': 'Right. What would the overall plain-English steps look like,

{'qid': 5000012,
 'scenario': 1,
 'question': "Nancy is filling an aquarium for her fish. She fills it halfway and goes to answer the door. While she's gone, her cat knocks the aquarium over and spills half the water in it. Then Nancy comes back and triples the amount of water in the aquarium. If the aquarium is 4 feet long, 6 feet wide, and 3 feet high, how many cubic feet of water are in the aquarium?",
 'ground_truth': "First calculate the volume of the aquarium by multiplying its length, width and height: 4 ft * 6 ft * 3 ft = 72 cubic ft\nThen figure out what proportion of the aquarium is full after the cat knocks it over: 1/2 * 1/2 = 1/4\nThen figure out what proportion of the aquarium is full after Nancy refills it: 3 * 1/4 = 3/4\nNow multiply the proportion of the aquarium that's full by the aquarium's volume to find out how much water is in it: 72 cubic ft * 3/4 = 54 cubic ft\n 54",
 'student_incorrect_solution': 'The aquarium has a volume of 4 x 6 x 3 = 72 cubic feet.\nWhen Na

Вся предобработка будет связана исключительно со вторым датасетом, потому что первый на этапе генерации строго контролировался вручную и предобработки не требует.<br>
Всего будет отобрано 150 диалогов длинны больше 11, будет произведен парсинг в формат первого датасета, а так же оба датасета будут объединены в один финальный.<br>
Так же не будет никакой валидации и проверок диалогов, потому что все было просмотрено и проверенно вручную.

In [7]:
MIN_DIALOG_LENGTH=11

filtered_sft_df2 = sft_df2['train'].filter(
    lambda dialog: len(dialog['conversation'].split('|EOM|'))>MIN_DIALOG_LENGTH
)
len(filtered_sft_df2)

Filter:   0%|          | 0/2262 [00:00<?, ? examples/s]

1123

In [8]:
SEED=42
N_DIALOGS=150

def preprocessing_sft_df2(data):
  dialog = data['conversation'].split('|EOM|')
  ans=list()

  for s in dialog:
    if s.startswith('Teacher'):
      ans.append(
          {
              'role':'tutor',
              'content':s.replace('Teacher: (probing)', '').replace('Teacher: (generic)', '').replace('Teacher: (focus)', '').replace('Teacher: (telling)', '').strip()
          }
      )
    else:
      ans.append(
          {
              'role':'student',
              'content':s.replace('Student: ', '').strip()
          }
      )
  return { 'messages':ans }

filtered_sft_df2 = filtered_sft_df2.shuffle(seed=SEED).select(range(N_DIALOGS)).map(
    preprocessing_sft_df2
)
display(filtered_sft_df2[0])

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'qid': 5000266,
 'scenario': 1,
 'question': 'Jake can wash his car with 1 bottle of car wash soap 4 times.  If each bottle costs $4.00, and he washes his car once a week for 20 weeks, how much does he spend on car soap?',
 'ground_truth': '1 bottle of soap will last for 4 washes and he needs enough bottles for 20 weeks so 20/4 = 5 bottles\nEach bottle cost $4.00 and he needs 5 bottles so he will spend $4*5 = $20.00 on car soap\n 20',
 'student_incorrect_solution': 'Jake can wash his car 4 x 1 = 4 times with 1 bottle of car wash soap.\nIf each bottle costs $4.00, then each wash costs $4.00/1 = $4.00.\nOver 20 weeks, Jake washes his car 20 x 1 = 20 times.\nSo he will spend 20 x $4.00 = $80.00 on car soap over 20 weeks.\n 80.00',
 'student_profile': 'Claire is a 7th grade student. She struggle most with understanding what the problem is asking them to do.',
 'teacher_described_confusion': 'Student could not understand that the bottle lasted for 4 washes, and thought each wash cost $4',


In [9]:
schema = sft_df1['train'].select_columns('messages').features

sft_df = concatenate_datasets(
    [
        sft_df1['train'].select_columns('messages'),
        filtered_sft_df2.select_columns('messages').cast(schema)
    ]
)
display(sft_df[0])

Casting the dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

{'messages': [{'role': 'student',
   'content': 'i need to write a program that checks if a word is a palindrome and i just froze staring at a blank editor'},
  {'role': 'tutor',
   'content': "Let's not write any code yet. In plain English, what does it mean for a word to be a palindrome?"},
  {'role': 'student', 'content': 'it reads the same forwards and backwards'},
  {'role': 'tutor',
   'content': "Given that definition, what two things could you compare to check if that's true?"},
  {'role': 'student',
   'content': 'the original word, and some reversed version of it'},
  {'role': 'tutor',
   'content': 'Right. What would the overall plain-English steps look like, before any actual code?'},
  {'role': 'student',
   'content': 'take the word, reverse it somehow, then compare the reversed version to the original'},
  {'role': 'tutor',
   'content': "That's a complete plan. Is there anything about capitalization or spaces in the input you might need to handle before comparing?"},
  

In [10]:
sft_df.save_to_disk('/content/drive/MyDrive/projects/datasets/sft_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

После предобработки `mathdial` и объединения датасетов, я посмотрел распределение токенов, для того чтобы в последующем правильно задать `max_sequence_length` и случайно не обрезать половину диалогов.<br>
Дополнительно я провалидировал датасет к стандартному шаблону `assistent`-`user`.

In [11]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [12]:
role = {'tutor':'assistant', 'student':'user'}

sft_df = sft_df.map(
    lambda dialog: {'messages': [{'role':role.get(message['role']), 'content': message['content']} for message in dialog['messages']]}
)

sft_df[0]['messages']

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

[{'role': 'user',
  'content': 'i need to write a program that checks if a word is a palindrome and i just froze staring at a blank editor'},
 {'role': 'assistant',
  'content': "Let's not write any code yet. In plain English, what does it mean for a word to be a palindrome?"},
 {'role': 'user', 'content': 'it reads the same forwards and backwards'},
 {'role': 'assistant',
  'content': "Given that definition, what two things could you compare to check if that's true?"},
 {'role': 'user',
  'content': 'the original word, and some reversed version of it'},
 {'role': 'assistant',
  'content': 'Right. What would the overall plain-English steps look like, before any actual code?'},
 {'role': 'user',
  'content': 'take the word, reverse it somehow, then compare the reversed version to the original'},
 {'role': 'assistant',
  'content': "That's a complete plan. Is there anything about capitalization or spaces in the input you might need to handle before comparing?"},
 {'role': 'user',
  'cont

In [13]:
tokenization_df = sft_df.map(
    lambda dialog: {'tokens_length': len(tokenizer.apply_chat_template(dialog['messages'], tokenize=True, return_dict=True)['input_ids'])}
)

lengths = tokenization_df['tokens_length']

print(f'Средняя длинна: {sum(lengths) / len(lengths):.3f}\nМаксимальная длинна: {max(lengths):.3f}')
print('Длинна по прецентилям:')
print(
    f'p50: {np.percentile(lengths, 50):.3f}\np90: {np.percentile(lengths, 90):.3f}\np90: {np.percentile(lengths, 90):.3f}\np99: {np.percentile(lengths, 99):3f}'
)

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Средняя длинна: 369.618
Максимальная длинна: 2518.000
Длинна по прецентилям:
p50: 296.000
p90: 713.400
p90: 713.400
p99: 1350.090000


По итогу я взял максимальную длинну 2518, чтобы вообще не было обрезки диалогов: они и так недлинные, поэтому никаких ограничений по памяти не будет.<br>
Финальным этапом я разделил датасет на обучающую и валидационную выборки, а так же провалидировал в формат модели `Qwen`, используя токенизатор.

In [14]:
split_df = sft_df.train_test_split(test_size=0.15, seed=SEED)

train_df, val_df = split_df['train'], split_df['test']

train_df = train_df.map(
    lambda dialog : {'text' : tokenizer.apply_chat_template(dialog['messages'], tokenize=False, add_generation_prompt=False)}
)

val_df = val_df.map(
    lambda dialog : {'text' : tokenizer.apply_chat_template(dialog['messages'], tokenize=False, add_generation_prompt=False)}
)

Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

# **Настройка параметров дообучения и само дообучение**

Теперь предстоит главный этап - дообучение модели `Qwen2.5-1.5B-Instruct` быть тьютером.<br>
Для загрузки модели я использовал фреймворк `Unsloth`, предоставляющий заранее пропатченные Triton-ядрами модели. Методом дообучения я выбрал `LoRA`, потому что из-за маленького веса модели она спокойно поместится в Tesla T4 (14.5GB).<br>
Весь процесс обучения я буду логировать в сервис `Weights & Bias` (`wandb`), для отслеживания затрат GPU и графиков обучения.


In [15]:
MODEL_NAME='Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN=2500

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name=MODEL_NAME,      # имя модели
  max_seq_length=MAX_SEQ_LEN, # параметр для настройки RoPE-scaling и выделения KV-cache
  dtype=None,                 # fp16/bf16 (будет настроен в самом trainer)
  load_in_4bit=False          # NF-4 квантизация для QLoRA
)

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
model = FastLanguageModel.get_peft_model(
    model,                                # сама модель
    r=16,                                 # размерность A, B матриц: "емкость адаптера"
    lora_alpha=32,                        # коэффициент масштабирования: реальный вклад адаптера
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],                                    # на какие слои будут навешены адаптеры
    lora_dropout=0.1,                     # регуляризация самих адапторов
    bias='none',                          # обучать/не обучать bias отдельно
    use_gradient_checkpointing='unsloth'  # пересчет активаций вместо хранения, для экономии памяти
)

Так же после настройки trainer я указал, что обучаться модель будет только на сообщениях ассистента, иначе она просто будет учить реплики ученика.

In [21]:
training_args = SFTConfig(
    run_name='qwen2.5-1.5B-tutor_sft',    # имя запуска обучения (для wandb)
    output_dir='./output',                # локальная директория для сохранения чекпоинтову модели
    per_device_train_batch_size=4,        # сколько диалогов обрабатывается за один проход на одном GPU одновременно
    gradient_accumulation_steps=3,        # кол-во шагов накопления градиентов (чтобы не обновлять после каждого батча)
    warmup_ratio=0.1,                     # доля от общего кол-ва шагов обучения, в течение которого растет learning_rate
    num_train_epochs=4,                   # кол-во прохождения обучающей выборке через модель
    learning_rate=2e-4,                   # величина шага обновления весов на каждой иттерации: насколько сильно двигаются параметры в адаптере
    fp16=True,                            # обучение в 16-битной плавающей точкой (bf16 не поддерживается на T4)
    logging_steps=10,                     # как часто будут писаться метрики в wandb
    save_strategy='epoch',                # когда модель сохраняет чекпоинт на диск (можно пв конца каждого шага)
    save_total_limit=3,                   # сколько последних чекпоинтов хранить на диске
    eval_strategy='epoch',                # когда прогонять модель по на валидационной выборке
    load_best_model_at_end=True,          # автоматически подгружать лучший чекпоинт по завершению обучения
    metric_for_best_model="eval_loss",    # какую метрику использовать для сравнения чекпоинтов
    optim='adamw_torch',                  # какую реализацию оптимизатора использовать
    lr_scheduler_type='cosine',           # форма кривой снижения learning_rate
    seed=SEED,                            # генератор случайных чисел
    report_to=['wandb'],                  # подключение wandb (куда отправлять логи)
    dataset_text_field='text',            # название колонки откуда брать текст
    max_length=MAX_SEQ_LEN,               # максимальная длина последовательности токенов
    packing=False                         # упаковка нескольких примеров в одну последовательность
)

trainer = SFTTrainer(
    model=model,              # сама модель
    tokenizer=tokenizer,      # сам токенизатор
    train_dataset=train_df,   # обучающая выборка
    eval_dataset=val_df,      # валидационная выборка
    args=training_args,       # настройка
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/552 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/98 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.653505,1.724881
2,1.387273,1.517263
3,1.048373,1.506323
4,0.700459,1.579853


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=184, training_loss=1.2679232903148816, metrics={'train_runtime': 1129.2084, 'train_samples_per_second': 1.955, 'train_steps_per_second': 0.163, 'total_flos': 1.1001519882080256e+16, 'train_loss': 1.2679232903148816, 'epoch': 4.0})

# **Оценка результатов**

Во время обучения я следил за графиками в `wandb`. По итогу ошибка модели на обучающей выборке монотонно убывала в ходе всего обучения, а на валидационной ошибка убывала до 3-й эпохи, где она достигла своего минимума, а после немного выросла на 4-й. Это классическое переобучение модели, что и следовало ожидать на таком маленьком датасете. Так как я автоматически задал ставить итоговой моделью лучшую из всех чекпоинтов, то переобучение никак не повлияло на финал, потому что была взята модель из чекпоинта 3-й эпохи.<br>
Итоговая перплексия модели составила 4.51, что ожидаемо после дообучения на небольшом и качественном датасете.<br>
Я не стал проводить глубокую оценку (к примеру сравнения сгенерированных диалогов или LLM-as-judge) на этом этапе (будет сделано по итогам всей системы). Здесь же достаточно количественных метрик, чтобы подвердить успешность дообучения.

In [24]:
print(f'Лучший чекпоинт: {trainer.state.best_model_checkpoint}')
print(f'Перплексия: {math.exp(trainer.state.best_metric):.2f}')

Лучший чекпоинт: ./output/checkpoint-138
Перплексия: 4.51


# **Выгрузка результатов**

Итоговую модель я смержил с основной и выгрузил на HuggingFace в формате merged 16-бит весов.

In [25]:
model.save_pretrained_merged('qwen2.5-1.5b-exam-tutor', tokenizer, save_method='merged_16bit')
model.push_to_hub_merged('ptvnck/qwen2.5-1.5b-exam-tutor', tokenizer, save_method='merged_16bit')

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `qwen2.5-1.5b-exam-tutor`: 100%|██████████| 1/1 [00:52<00:00, 52.40s/it]


Successfully copied all 1 files from cache to `qwen2.5-1.5b-exam-tutor`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:13<00:00, 73.15s/it]


Unsloth: Merge process complete. Saved to `/content/qwen2.5-1.5b-exam-tutor`


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...m-tutor/model.safetensors:   1%|          | 15.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:58<00:00, 118.34s/it]


Unsloth: Merge process complete. Saved to `/content/ptvnck/qwen2.5-1.5b-exam-tutor`
